# Fig. 1 -- model, flippability, and accessible subspaces

Each panel is drawn as its own figure at a fixed physical scale (points
per data unit) and saved to `figures/fig-01/<panel>.{pdf,svg,png}`
without its letter. The assembly step arranges the panels and adds the
letters. All sizes below are in points unless named otherwise.

In [ ]:
import itertools

import colorcet as cc
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import networkx as nx
import numpy as np

from qising_networks import FIGURES_DIR, PROJECT_ROOT
from qising_networks.model import degree_to_h_on_off, generate_all_ising_states, h_on_off_to_h, interaction_to_hamiltonian
from qising_networks import fig_maker as fm
from qising_networks import fig_maker_helper as fmh

mpl.rc_file(PROJECT_ROOT / "matplotlibrc")
OUT = FIGURES_DIR / "fig-01"

FONTSIZE = 7
FERRO_COLOR = cc.cm.CET_D3_r(1.0)  # J > 0, as the network plots
HIGHLIGHT = "#d95f02"              # floppy (b, d) and flippable (c, e, f) spins
GAP_PT = 0.7                       # node edge to field-arrow tip

# Panel a: the 8-spin example network.
SCALE_A = 15.8                     # points per data unit
NODE_S_A = 166.0                   # scatter area; true radius sqrt(s)/4
EDGE_LW_A = 1.0
H_LEN_A = 9.5                      # field arrow length
H_SHAFT_A = 0.88                   # shaft width per unit of h

# Panels b-f: the small 3- and 5-spin diagrams share one node style.
NODE_S = 81.0
EDGE_LW = 0.7
H_LEN = 7.5
SCALE_BC = 18.0
SCALE_DE = 20.3
NUM_SPINS_MINI = 5
SPACING_MINI = 0.55
ARROW_LW = 0.8
TITLE_GAP = 3.0                    # arrow shaft to title baseline


def canvas(xlim, ylim, scale):
    """Figure whose single axes spans exactly xlim x ylim at `scale`
    points per data unit, so physical sizes are known before drawing."""
    w = (xlim[1] - xlim[0]) * scale / 72
    h = (ylim[1] - ylim[0]) * scale / 72
    fig = plt.figure(figsize=(w, h), layout=None)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")
    return fig, ax


def save(fig, name):
    fm.save_figure(fig, OUT, name, formats=("pdf", "svg", "png"),
                   bbox_inches="tight")
    plt.close(fig)


def node_radius(node_s):
    return np.sqrt(node_s) / 4


def arc(ax, xa, xb, y=0.0, lw=EDGE_LW, color=FERRO_COLOR):
    r = abs(xb - xa) / 2
    ax.add_patch(mpatches.Arc(((xa + xb) / 2, y), 2 * r, 2 * r, theta1=0,
                              theta2=180, lw=lw, color=color, zorder=1))


def node(ax, x, y, node_s, face, edge="k", ew=0.4):
    ax.scatter(x, y, s=node_s, facecolor=face, edgecolor=edge,
               linewidth=ew, zorder=4)


def h_arrow(ax, x, y, gap, length, shaft):
    """Upward field arrow below the node at (x, y), sized in points."""
    head_w = 3 * shaft
    head_l = min(1.5 * head_w, 0.35 * length)
    trans = (mtransforms.Affine2D().scale(1 / 72) + ax.figure.dpi_scale_trans
             + mtransforms.ScaledTranslation(x, y, ax.transData))
    ax.add_patch(mpatches.FancyArrow(
        0, -(gap + length), 0, length, width=shaft, head_width=head_w,
        head_length=head_l, length_includes_head=True, fc="k", lw=0,
        zorder=3, transform=trans, clip_on=False))


def open_arrow(ax, x0, x1, y, head_len, head_wid, both=False, **kw):
    """Horizontal line with open chevron head(s); head sizes in data units."""
    kw = dict(color="k", lw=ARROW_LW, solid_joinstyle="miter",
              solid_capstyle="butt", zorder=5) | kw
    ax.plot([x0, x1], [y, y], **kw)
    ax.plot([x1 - head_len, x1, x1 - head_len],
            [y + head_wid, y, y - head_wid], **kw)
    if both:
        ax.plot([x0 + head_len, x0, x0 + head_len],
                [y + head_wid, y, y - head_wid], **kw)


def mini_chain(ax, x0, edges, spins, h, scale, faces=None, mask=None,
               shrink=1.0, y=0.0):
    """5-spin chain of panels d-f: arcs above, field arrows below; `mask`
    outlines the floppy/flippable spins. `shrink` scales every physical
    size (panel f's insets)."""
    xs = x0 + np.arange(len(spins)) * SPACING_MINI
    node_s = NODE_S * shrink**2
    for i, j in edges:
        arc(ax, xs[i], xs[j], y, lw=EDGE_LW * shrink)
    for k, (xk, s) in enumerate(zip(xs, spins)):
        h_arrow(ax, xk, y, node_radius(node_s) + GAP_PT * shrink,
                H_LEN * shrink, EDGE_LW * shrink * h[k])
        face = faces[k] if faces is not None else ("k" if s > 0 else "white")
        hl = mask is not None and mask[k]
        node(ax, xk, y, node_s, face, HIGHLIGHT if hl else "k",
             (1.1 if hl else 0.35) * shrink)
    return xs


def random_connected_graph(n, p, seed):
    rng = np.random.default_rng(seed)
    while True:
        G = nx.gnp_random_graph(n, p, seed=int(rng.integers(0, 2**31)))
        if nx.is_connected(G):
            return G

## Example networks (panels d-f)

In [ ]:
def floppiness_examples(seed_graph=36, seed_h=112, n_examples=4):
    """One ferromagnetic network and field; spin configurations chosen to
    span floppiness 0 to 1. Floppy spins have zero local field."""
    G = random_connected_graph(NUM_SPINS_MINI, 0.5, seed_graph)
    edges, adj = sorted(G.edges()), nx.to_numpy_array(G)
    h = np.random.default_rng(seed_h).integers(1, 3, NUM_SPINS_MINI).astype(float)
    configs = np.array(list(itertools.product([-1, 1], repeat=NUM_SPINS_MINI)))
    n_floppy = np.sum(np.abs(configs @ adj.T + h) < 1e-9, axis=1)
    examples, masks, values = [], [], []
    for target in np.round(np.linspace(0, NUM_SPINS_MINI, n_examples)).astype(int):
        s = configs[int(np.argmin(np.abs(n_floppy - target)))]
        examples.append((edges, s, h))
        masks.append(np.abs(s @ adj.T + h) < 1e-9)
        values.append(masks[-1].sum() / NUM_SPINS_MINI)
    return examples, masks, values


def flippability_examples(seed_graph=101, seed_h=202, n_examples=4):
    """Independent ferromagnetic networks whose fields make exactly 0, 2,
    3, 5 of the 5 spins flippable ((degree + h) even)."""
    graph_rng = np.random.default_rng(seed_graph)
    h_rng = np.random.default_rng(seed_h)
    examples, masks, values, adjs = [], [], [], []
    for target in np.round(np.linspace(0, NUM_SPINS_MINI, n_examples)).astype(int):
        g_seed, k_seed = (int(graph_rng.integers(0, 2**31)),
                          int(h_rng.integers(0, 2**31)))
        G = random_connected_graph(NUM_SPINS_MINI, 0.5, g_seed)
        adj = nx.to_numpy_array(G)
        deg = (adj > 0).sum(axis=1).astype(int)
        h = h_on_off_to_h(*degree_to_h_on_off(deg), int(target),
                             np.random.default_rng(k_seed))
        examples.append((sorted(G.edges()), np.ones(NUM_SPINS_MINI, int), h))
        masks.append((deg + h.astype(int)) % 2 == 0)
        values.append(target / NUM_SPINS_MINI)
        adjs.append(adj)
    return examples, masks, values, adjs


floppy_examples, floppy_masks, floppy_values = floppiness_examples()
flip_examples, flip_masks, flip_values, flip_adjs = flippability_examples()

## a: the model on an 8-spin network

In [ ]:
def make_panel_data(graph_seed=8, spin_seed=42, h_seed=42, n=8):
    G = random_connected_graph(n, 0.35, graph_seed)
    spins = np.random.default_rng(spin_seed).choice([-1, 1], size=n)
    h = np.random.default_rng(h_seed).integers(1, 3, size=n)
    return sorted(G.edges()), spins, h


edges, spins, h = make_panel_data()
x = np.arange(len(spins)) * 1.4
fig, ax = canvas((-2.96, 10.4), (-1.35, 3.5), SCALE_A)
r_a = node_radius(NODE_S_A)
for i, j in edges:
    arc(ax, x[i], x[j], lw=EDGE_LW_A)
for xi, si, hi in zip(x, spins, h):
    h_arrow(ax, xi, 0, r_a + GAP_PT, H_LEN_A, H_SHAFT_A * hi)
    node(ax, xi, 0, NODE_S_A, "k" if si > 0 else "white")
# Transverse field: dashed arrow into the chain from the left.
tip, tail = -0.95 * 1.4, -1.9 * 1.4
head_l, head_hw = 3.15 / SCALE_A, 1.05 / SCALE_A
ax.plot([tail, tip - head_l], [0, 0], color="k", lw=0.7, ls="dashed", zorder=3)
ax.add_patch(mpatches.Polygon([[tip, 0], [tip - head_l, head_hw],
                               [tip - head_l, -head_hw]], fc="k", lw=0))
ax.text((tail + tip) / 2, 0.1, r"$\Gamma = 0.1J$", ha="center", va="bottom",
        fontsize=FONTSIZE)
ax.text(x[-1], -(r_a + GAP_PT + H_LEN_A) / SCALE_A - 0.25,
        r"$h_i \in \{+J,\,+2J\}$", ha="center", va="top", fontsize=FONTSIZE)
i, j = max(edges, key=lambda e: x[e[1]] - x[e[0]])
ax.text((x[i] + x[j]) / 2, (x[j] - x[i]) / 2 + 0.08,
        r"$J_{i,j} \in \{0,\,+J\}$", ha="center", va="bottom", fontsize=FONTSIZE)
ax.legend(handles=[mpl.lines.Line2D([], [], ls="none", marker="o",
                                    markersize=2 * r_a, mfc=fc, mec="k", mew=0.4)
                   for fc in ("k", "white")],
          labels=[r"$\uparrow$", r"$\downarrow$"], loc="upper left",
          frameon=False, fontsize=FONTSIZE, handlelength=1.0,
          handletextpad=0.4, borderaxespad=0.0)
save(fig, "a")

## b, c: floppy vs rigid, flippable vs unflippable

Each row shows one spin before and after its flip, with h = 2J. Two
ferromagnetic neighbours that are both down cancel the field (floppy);
one up neighbour leaves 2J (rigid). With a single neighbour the local
field is J or 3J for every neighbour configuration (unflippable).

In [ ]:
R = node_radius(NODE_S)
Y_BOTTOM = -(R + GAP_PT + H_LEN) / SCALE_BC - 0.12
ROW_YLIM = (Y_BOTTOM, 0.9)


def concept_row(ax, title, bonds, neighbours, highlight, negated):
    """Two copies of a 3-site chain (focal spin up, then down) joined by a
    double arrow; `bonds` are (i, j) pairs, `neighbours` the faces of
    sites 0 and 2 (None: site absent)."""
    for xs, focal_up in [(np.array([0.0, 1, 2]), True),
                         (np.array([4.0, 5, 6]), False)]:
        for i, j in bonds:
            arc(ax, xs[i], xs[j])
        for k in (0, 2):
            if neighbours[k // 2] is not None:
                node(ax, xs[k], 0, NODE_S, neighbours[k // 2], ew=0.4)
        node(ax, xs[1], 0, NODE_S, "k" if focal_up else "white",
             HIGHLIGHT if highlight else "k", 1.1 if highlight else 0.4)
        h_arrow(ax, xs[1], 0, R + GAP_PT, H_LEN, 2 * EDGE_LW)
    open_arrow(ax, 2.3, 3.7, 0, 4.5 / SCALE_BC, 2.5 / SCALE_BC, both=True)
    if negated:
        ax.plot([2.78, 3.22], [-0.22, 0.22], color="k", lw=1.3,
                solid_capstyle="butt", zorder=6)
    ax.text(3.0, 0.9, title, fontsize=FONTSIZE, ha="center", va="top")


def concept_panel(rows):
    """Two stacked concept rows in one figure."""
    w = 6.8 * SCALE_BC / 72
    row_h = (ROW_YLIM[1] - ROW_YLIM[0]) * SCALE_BC / 72
    gap = 3 / 72
    fig = plt.figure(figsize=(w, 2 * row_h + gap), layout=None)
    for k, row in enumerate(rows):
        ax = fig.add_axes([0, (1 - k) * (row_h + gap) / (2 * row_h + gap),
                           1, row_h / (2 * row_h + gap)])
        ax.set_xlim(-0.4, 6.4)
        ax.set_ylim(*ROW_YLIM)
        ax.axis("off")
        concept_row(ax, *row)
    return fig


both = [(0, 1), (1, 2)]
save(concept_panel([
    ("Floppy", both, ("white", "white"), True, False),
    ("Rigid", both, ("white", "k"), False, True),
]), "b")
save(concept_panel([
    ("Flippable", both, ("gray", "gray"), True, False),
    ("Unflippable", [(0, 1)], ("gray", None), False, True),
]), "c")

## d, e: floppiness and flippability spectra

In [ ]:
def spectrum_panel(title, examples, values, masks, faces=None):
    mini_w = (NUM_SPINS_MINI - 1) * SPACING_MINI
    stride = mini_w + 0.9
    total = (len(examples) - 1) * stride + mini_w
    y_value = mini_w / 2 + 0.17
    y_arrow = y_value + 0.37
    y_bottom = -(R + GAP_PT + H_LEN) / SCALE_DE - 0.2
    y_top = y_arrow + (TITLE_GAP + FONTSIZE) / SCALE_DE
    fig, ax = canvas((-0.3, total + 0.3), (y_bottom, y_top), SCALE_DE)
    for k, (edges, spins, h) in enumerate(examples):
        mini_chain(ax, k * stride, edges, spins, h, SCALE_DE, faces=faces,
                   mask=masks[k])
        ax.text(k * stride + mini_w / 2, y_value, f"{values[k]:.2g}",
                fontsize=FONTSIZE, ha="center", va="bottom")
    open_arrow(ax, 0, total, y_arrow, 0.12, 0.09)
    ax.text(total / 2, y_arrow + TITLE_GAP / SCALE_DE, title,
            fontsize=FONTSIZE, ha="center", va="baseline")
    return fig


save(spectrum_panel(r"Floppiness $f$", floppy_examples, floppy_values,
                    floppy_masks), "d")
save(spectrum_panel(r"Flippability $\phi$", flip_examples, flip_values,
                    flip_masks, faces=["gray"] * NUM_SPINS_MINI), "e")

## f: Hilbert-space diagrams and Hamiltonians of panel e's networks

Every Fock state of each network, projected with one shared weighted PCA
(fitted on the flippability-1 network) and coloured by its classical
energy. Black edges join states of equal energy, i.e. the flips the
transverse field allows; the matrix below is the Hamiltonian ordered by
energy sector (solid frames) and accessible subspace (dashed frames).

In [ ]:
def hilbert_diagram(ax, wpca, hyp_adj, hyp_same, energies, norm,
                    node_s=48, lw=0.3):
    states = generate_all_ising_states(int(np.log2(len(hyp_adj))))
    xy = wpca.transform(states)[:, :2]
    for i, j in zip(*np.nonzero(np.triu(hyp_adj))):
        same = hyp_same[i, j]
        ax.plot(xy[[i, j], 0], xy[[i, j], 1], color="k" if same else "0.75",
                lw=2 * lw if same else lw, alpha=1 if same else 0.6,
                zorder=2 if same else 1)
    # Least-connected states first, so the allowed web stays on top.
    comp_size = np.zeros(len(hyp_same), int)
    for comp in nx.connected_components(nx.from_numpy_array(hyp_same)):
        comp_size[list(comp)] = len(comp)
    order = np.lexsort((comp_size, hyp_same.sum(axis=1)))
    ax.scatter(xy[order, 0], xy[order, 1], s=node_s, c=energies[order],
               cmap=cc.cm.coolwarm, norm=norm, zorder=3)
    return xy


def block_order(hyp_same, energies):
    """Basis order: energy sectors ascending, each split into its
    accessible subspaces (largest first). Returns the order and the
    (start, size) of every sector and subspace."""
    G = nx.from_numpy_array(hyp_same)
    order, sectors, subspaces = [], [], []
    for E in np.unique(energies):
        members = np.nonzero(energies == E)[0]
        comps = sorted((sorted(c) for c in
                        nx.connected_components(G.subgraph(members))),
                       key=lambda c: (-len(c), c[0]))
        sectors.append((len(order), len(members)))
        for c in comps:
            subspaces.append((len(order), len(c)))
            order.extend(c)
    return np.array(order), sectors, subspaces


def hamiltonian_matrix(ax, H, order, sectors, subspaces, norm, lw=0.5):
    Hp = H[np.ix_(order, order)]
    ax.imshow(Hp, cmap=cc.cm.coolwarm, norm=norm, interpolation="nearest",
              aspect="auto")
    # Transverse-field entries are ~1 % of the energy range: paint black.
    overlay = np.zeros(Hp.shape + (4,))
    overlay[(Hp != 0) & ~np.eye(len(Hp), dtype=bool)] = (0, 0, 0, 1)
    ax.imshow(overlay, interpolation="nearest", aspect="auto", zorder=2)
    ax.set_axis_off()
    for (start, size), ls, d in [(s, "solid", 0) for s in sectors] + \
                                [(s, (0, (1.5, 1.0)), 0.25) for s in subspaces]:
        if size > 1:
            ax.add_patch(mpatches.Rectangle(
                (start - 0.5 + d, start - 0.5 + d), size - 2 * d, size - 2 * d,
                fill=False, ec="k", lw=lw, ls=ls, zorder=3))


def ket(state):
    arrows = "".join(r"\uparrow" if s > 0 else r"\downarrow" for s in state)
    return rf"$\mid {arrows}\rangle$"


hilbert = []
for (_, _, h), adj in zip(flip_examples, flip_adjs):
    _, hyp, hyp_same, en = fmh.hilbert_space_projection(adj, h, seed=42)
    hilbert.append((hyp, hyp_same, np.asarray(en).ravel()))
wpca, _, _, _ = fmh.hilbert_space_projection(flip_adjs[-1], flip_examples[-1][2],
                                          seed=42)
norm = mcolors.CenteredNorm(0.0, max(np.abs(en).max() for *_, en in hilbert))

# Layout in points from the top-left corner: arrow strip, diagrams,
# matrices, colorbar beside the diagrams.
W, H = 495.0, 252.0
COL_X, COL_W = [0.0, 123.18, 246.35, 369.53], 107.61
DIAG_Y, DIAG_H = 33.6, 100.8
MAT_Y, MAT = 144.5, 106.2
ARROW_Y, ARROW_X1 = 16.0, 477.1
fig = plt.figure(figsize=(W / 72, H / 72), layout=None)


def add_axes_pt(x, y, w, h):
    return fig.add_axes([x / W, 1 - (y + h) / H, w / W, h / H])


over = add_axes_pt(0, 0, W, H)  # annotation layer in points, y downward
over.set_xlim(0, W)
over.set_ylim(H, 0)
over.axis("off")
open_arrow(over, 0, ARROW_X1, ARROW_Y, 2.47, -1.85)
over.text(ARROW_X1 / 2, ARROW_Y - TITLE_GAP, r"Flippability $\phi$",
          fontsize=FONTSIZE, ha="center", va="baseline")
for k, (hyp, hyp_same, en) in enumerate(hilbert):
    over.text(COL_X[k] + COL_W / 2, ARROW_Y + 7, f"{flip_values[k]:.2g}",
              fontsize=FONTSIZE, ha="center", va="top")
    ax = add_axes_pt(COL_X[k], DIAG_Y, COL_W, DIAG_H)
    xy = hilbert_diagram(ax, wpca, hyp, hyp_same, en, norm)
    ax.set_aspect("equal")
    ax.axis("off")
    pad = 0.02 * np.ptp(xy, axis=0)
    ax.set_xlim(xy[:, 0].min() - pad[0], xy[:, 0].max() + pad[0])
    ax.set_ylim(xy[:, 1].min() - pad[1], xy[:, 1].max() + pad[1])
    if k == 0:
        # Two neighbouring Fock states, to show what a node is.
        states = generate_all_ising_states(NUM_SPINS_MINI)
        bottom = int(np.argmin(xy[:, 1]))
        for idx in (bottom, bottom ^ (1 << (NUM_SPINS_MINI - 2))):
            ax.annotate(ket(states[idx]), xy=xy[idx], xytext=(24, 0),
                        textcoords="offset points", fontsize=FONTSIZE,
                        ha="left", va="center",
                        arrowprops=dict(arrowstyle="->", lw=0.4, color="0.3",
                                        shrinkA=0, shrinkB=4))
    # Source network as an inset in the diagram's top-left corner.
    inset = add_axes_pt(COL_X[k] - 2.3, DIAG_Y - 9.7, 31.7, 20.7)
    inset.set_xlim(-0.3, 2.5)
    inset.set_ylim(-0.61, 1.22)
    inset.axis("off")
    edges, spins, h = flip_examples[k]
    mini_chain(inset, 0, edges, spins, h, 11.3, faces=["gray"] * 5,
               mask=flip_masks[k], shrink=0.55)
    # Hamiltonian in the diagram's basis order (interaction_to_hamiltonian
    # counts bit 0 as up, the reverse of generate_all_ising_states).
    Hk = interaction_to_hamiltonian(flip_adjs[k], np.full(5, 0.1),
                                       h)[::-1, ::-1]
    assert np.allclose(np.diag(Hk), en)
    hamiltonian_matrix(add_axes_pt(COL_X[k] + 0.7, MAT_Y, MAT, MAT), Hk,
                       *block_order(hyp_same, en), norm)
cax = add_axes_pt(484.3, DIAG_Y, 7.4, DIAG_H)
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm, cc.cm.coolwarm), cax=cax)
cbar.set_label(r"Classical energy $H(\boldsymbol{s})$")
cbar.outline.set_visible(False)
save(fig, "f")